# ACNE04 Dataset Exploration & Pre-processing Pipeline

**Project Title:** Comparative Study of Transfer Learning Models for Acne Severity Classification on the ACNE04 Benchmark Dataset  
**Objective:** Inspect raw dataset files, understand class distributions, demystify naming conventions, verify image dimensions, and generate structured metadata (`acne04_metadata.csv`) with stratified train-val-test splits.

---

### Dataset Overview (ACNE04)
- **Origin:** Facial acne photographs captured in clinical dermatological settings.
- **Clinical Standard:** Hayashi Acne Severity Grading Criteria (4 classes: Level 0 to Level 3).
- **Image Native Resolution:** $1024 \times 1024$ pixels in RGB format.
- **Key Research Challenge:** Significant class imbalance across severity grades.

## 1. Setup & Environment Imports

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

import re
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split

# Set visual style for publication-ready figures
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

print("Libraries successfully imported!")

## 2. Directory Structure & Path Resolution

Let's define the paths pointing to `data/raw/acne_1024` and verify the directory structure.

In [ ]:
# Robust path configuration (works both from repo root or notebooks/ folder)
current_dir = Path(os.getcwd())
if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "acne_1024"
METADATA_DIR = PROJECT_ROOT / "data" / "metadata"
OUTPUT_FIG_DIR = PROJECT_ROOT / "outputs" / "figures"

METADATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root Directory : {PROJECT_ROOT}")
print(f"Raw Data Directory     : {RAW_DATA_DIR}")
print(f"Directory exists?      : {RAW_DATA_DIR.exists()}")

### 2.1 Raw Dataset Inventory & File Type Discovery

Before diving into modeling, let us inspect what exact file types and auxiliary files are stored in `data/raw/`.
Many researchers opening this dataset for the first time wonder:
- **What format are the images?** Standard JPEG (`.jpg`), 3-channel RGB, native 1024x1024 resolution.
- **What are the `.jsonl` files?** Prompt conditioning files created for diffusion/generative models (e.g. `metadata.jsonl`).
- **What are `all_1024` and `small_1024`?** Auxiliary folders (`all_1024` combines all classes without subfolders; `small_1024` is a 201-sample subset).
- **What is `sim_acne.csv`?** A simulated longitudinal acne clinical trials CSV table.
- **Which folders are used for our classification research?** The 4 canonical Hayashi grade folders: `acne0_1024`, `acne1_1024`, `acne2_1024`, and `acne3_1024`.

In [ ]:
# Inspect all files and extensions in data/raw
from collections import Counter

raw_parent = PROJECT_ROOT / 'data' / 'raw'
ext_counter = Counter()

for item in sorted(raw_parent.rglob('*')):
    if item.is_file():
        ext_counter[item.suffix.lower()] += 1

print('=== File Extension Breakdown in data/raw ===')
for ext, count in ext_counter.most_common():
    label = ext if ext else '<no extension>'
    print(f'  Extension: {label:12} | Count: {count:5} files')

print('\n=== Primary Subfolders in data/raw/acne_1024 ===')
if RAW_DATA_DIR.exists():
    for sub in sorted(RAW_DATA_DIR.iterdir()):
        if sub.is_dir():
            f_count = len(list(sub.glob('*')))
            sample = next(sub.glob('*'), None)
            sample_name = sample.name if sample else 'Empty'
            print(f'  Folder: {sub.name:22} | Total Items: {f_count:5} | Sample: {sample_name}')


## 3. Demystifying Dataset Folders & File Naming Conventions

The dataset is organized into multiple subfolders:
- `acne0_1024` : Grade 0 (Normal / Mildest)
- `acne1_1024` : Grade 1 (Mild)
- `acne2_1024` : Grade 2 (Moderate)
- `acne3_1024` : Grade 3 (Severe)
- `all_1024`   : Aggregated image collection

> **Crucial Filename Note:** 
> In the original distribution of the ACNE04 dataset, filenames use the pattern `levleX_Y.jpg` (notice the typographical error in "`levle`" instead of "`level`"). 
> Automated parsers must account for this spelling.

In [ ]:
# Severity Grade Mapping following Hayashi Criteria
GRADE_MAPPING = {
    "acne0_1024": {"label": 0, "severity": "Level 0 (Normal / Clear)", "hayashi": "0-5 Lesions"},
    "acne1_1024": {"label": 1, "severity": "Level 1 (Mild)", "hayashi": "6-20 Lesions"},
    "acne2_1024": {"label": 2, "severity": "Level 2 (Moderate)", "hayashi": "21-50 Lesions"},
    "acne3_1024": {"label": 3, "severity": "Level 3 (Severe)", "hayashi": ">50 Lesions"}
}

# Scan and inspect each class folder
folder_stats = []
for folder_name, meta in GRADE_MAPPING.items():
    folder_path = RAW_DATA_DIR / folder_name
    if folder_path.exists():
        files = list(folder_path.glob("*.jpg")) + list(folder_path.glob("*.jpeg"))
        sample_name = files[0].name if files else "None"
        folder_stats.append({
            "Folder": folder_name,
            "Class Label": meta["label"],
            "Severity Level": meta["severity"],
            "Clinical Criterion": meta["hayashi"],
            "Image Count": len(files),
            "Sample Filename": sample_name
        })

stats_df = pd.DataFrame(folder_stats)
display(stats_df)

## 4. Extracting Metadata & Verifying Image Integrity

We now iterate through all class folders to construct a comprehensive metadata dataframe containing:
- File path
- Filename
- Target integer label ($y \in \{0, 1, 2, 3\}$)
- Categorical severity name
- Resolution (width $\times$ height)
- Color channels
- File size in kilobytes

In [ ]:
records = []

print("Indexing images and extracting physical properties...")
for folder_name, meta in GRADE_MAPPING.items():
    folder_path = RAW_DATA_DIR / folder_name
    if not folder_path.exists():
        continue
        
    for file_path in folder_path.glob("*.jpg"):
        try:
            with Image.open(file_path) as img:
                width, height = img.size
                channels = len(img.getbands())
                mode = img.mode
            
            file_size_kb = round(os.path.getsize(file_path) / 1024, 2)
            
            # Extract level number from filename using regex
            match = re.search(r'levle?(\d+)_', file_path.name, re.IGNORECASE)
            detected_level = int(match.group(1)) if match else meta["label"]
            
            records.append({
                "relative_path": str(file_path.relative_to(PROJECT_ROOT)).replace("\\", "/"),
                "filename": file_path.name,
                "label": meta["label"],
                "severity": meta["severity"],
                "width": width,
                "height": height,
                "channels": channels,
                "color_mode": mode,
                "size_kb": file_size_kb
            })
        except Exception as e:
            print(f"Warning: Failed to load {file_path.name}: {e}")

df_metadata = pd.DataFrame(records)
print(f"Total images successfully indexed: {len(df_metadata)}")
df_metadata.head(8)

## 5. Class Distribution & Imbalance Analysis

Let's visualize the class distribution to evaluate the degree of imbalance.

In [ ]:
palette = ["#2ecc71", "#3498db", "#f39c12", "#e74c3c"]
class_counts = df_metadata["severity"].value_counts().loc[[
    "Level 0 (Normal / Clear)",
    "Level 1 (Mild)",
    "Level 2 (Moderate)",
    "Level 3 (Severe)"
]]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Bar Chart
sns.barplot(x=class_counts.index, y=class_counts.values, ax=axes[0], palette=palette)
axes[0].set_title("ACNE04 Class Frequency Distribution", fontsize=14, fontweight="bold", pad=12)
axes[0].set_ylabel("Number of Images", fontsize=12)
axes[0].set_xlabel("Severity Level", fontsize=12)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=15)

for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height())} ({p.get_height()/len(df_metadata)*100:.1f}%)",
                    (p.get_x() + p.get_width() / 2., p.get_height() + 10),
                    ha='center', va='bottom', fontsize=11, fontweight='bold')

# 2. Pie Chart
axes[1].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%', 
            startangle=140, colors=palette, explode=[0.03, 0.03, 0.05, 0.08],
            textprops={'fontsize': 11, 'fontweight': 'bold'})
axes[1].set_title("Proportional Class Share", fontsize=14, fontweight="bold", pad=12)

plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "class_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

print("Imbalance Ratio (Max / Min class):", round(class_counts.max() / class_counts.min(), 2))

### Observation & Methodological Implication
- **Imbalance Ratio:** The majority class (`Level 1`) has over **6x** more samples than the minority class (`Level 3`).
- **Implication for Training:** Standard Cross-Entropy loss will bias predictions towards Level 0 and Level 1. 
- **Recommended Mitigations:**
  1. Weighted Cross-Entropy Loss: $\mathcal{L} = -\sum w_c y_c \log(\hat{y}_c)$
  2. Focal Loss to focus on hard-to-classify samples.
  3. Balanced / Stratified Mini-Batch Sampling.
  4. Targeted Data Augmentation on Level 2 & Level 3 (CutMix, MixUp, ColorJitter).

## 6. Image Dimensions & Channel Consistency Verification

In [ ]:
print("Unique Image Dimensions (Width, Height):", df_metadata[["width", "height"]].drop_duplicates().to_dict(orient="records"))
print("Unique Color Modes:", df_metadata["color_mode"].unique())
print(f"File Size Range: Min = {df_metadata['size_kb'].min()} KB, Max = {df_metadata['size_kb'].max()} KB, Mean = {df_metadata['size_kb'].mean():.2f} KB")

## 7. Visual Sample Gallery Across Severity Levels

In [ ]:
num_samples_per_class = 3
fig, axes = plt.subplots(4, num_samples_per_class, figsize=(15, 16))

for row_idx, label in enumerate([0, 1, 2, 3]):
    class_subset = df_metadata[df_metadata["label"] == label]
    sample_records = class_subset.sample(n=min(num_samples_per_class, len(class_subset)), random_state=42)
    
    for col_idx, (_, row) in enumerate(sample_records.iterrows()):
        img_path = PROJECT_ROOT / row["relative_path"]
        with Image.open(img_path) as img:
            axes[row_idx, col_idx].imshow(img)
            axes[row_idx, col_idx].set_title(f"{row['severity']}\n{row['filename']}", fontsize=10)
            axes[row_idx, col_idx].axis("off")

plt.suptitle("ACNE04 Qualitative Visual Comparison Across Severity Grades", fontsize=16, fontweight="bold", y=0.99)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "sample_gallery.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Stratified Train-Validation-Test Splitting & Metadata Export

To ensure that all severity classes are proportionally represented in the evaluation benchmark, we perform a **Stratified 70% Train / 15% Validation / 15% Test Split**.

In [ ]:
# Step 1: Split into Train (70%) and Temp (30%)
train_df, temp_df = train_test_split(
    df_metadata, 
    test_size=0.30, 
    stratify=df_metadata["label"], 
    random_state=42
)

# Step 2: Split Temp into Validation (15%) and Test (15%)
val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.50, 
    stratify=temp_df["label"], 
    random_state=42
)

# Add 'split' column
df_metadata["split"] = "train"
df_metadata.loc[val_df.index, "split"] = "val"
df_metadata.loc[test_df.index, "split"] = "test"

# Display split summary
split_summary = pd.crosstab(df_metadata["severity"], df_metadata["split"], margins=True)
print("=== Stratified Dataset Split Summary ===")
display(split_summary)

# Export metadata index CSV
output_csv = METADATA_DIR / "acne04_metadata.csv"
df_metadata.to_csv(output_csv, index=False)
print(f"\nMetadata successfully saved to: {output_csv}")

## 9. Conclusion & Research Roadmap

### Key Takeaways:
1. **Uniform High Resolution:** All images are native $1024 \times 1024$ RGB. Resizing to standard backbone resolutions ($224 \times 224$ for CNNs/ViT or $384 \times 384$ for high-resolution ViT) will be fast and preserve key textural features.
2. **Ordinal Nature of Task:** Acne severity is an ordinal ranking ($0 < 1 < 2 < 3$). In addition to Multi-Class Accuracy and Macro-F1, evaluation should report **Mean Absolute Error (MAE)** to penalize distant misclassifications.
3. **Models to Benchmark:**
   - **MobileNetV2:** Lightweight baseline for edge/mobile clinical applications.
   - **EfficientNet-B0 / B2:** State-of-the-art parameter efficiency with compound scaling.
   - **ResNet-50:** Classical residual deep CNN benchmark.
   - **Vision Transformer (ViT-B/16):** Self-attention global context model for dermatological lesions.

**Next Step:** Proceed to model training pipeline in `src/train.py`.

## 10. PyTorch DataLoader Verification (Ready for Transfer Learning)

Let us verify that our generated metadata CSV seamlessly connects with the PyTorch `ACNE04Dataset` class and `DataLoader` in `src/data/dataset.py`.

In [ ]:
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import torch
from torch.utils.data import DataLoader
from src.data.dataset import ACNE04Dataset, get_default_transforms

# Load train and validation transforms
train_tf, eval_tf = get_default_transforms(image_size=224)

# Instantiate PyTorch Datasets
ds_train = ACNE04Dataset(metadata_csv=output_csv, split='train', transform=train_tf, root_dir=PROJECT_ROOT)
ds_val   = ACNE04Dataset(metadata_csv=output_csv, split='val', transform=eval_tf, root_dir=PROJECT_ROOT)
ds_test  = ACNE04Dataset(metadata_csv=output_csv, split='test', transform=eval_tf, root_dir=PROJECT_ROOT)

print(f'[PyTorch Dataset] Train samples : {len(ds_train)}')
print(f'[PyTorch Dataset] Val samples   : {len(ds_val)}')
print(f'[PyTorch Dataset] Test samples  : {len(ds_test)}')

# Test DataLoader batch extraction
loader = DataLoader(ds_train, batch_size=16, shuffle=True)
sample_images, sample_labels = next(iter(loader))

print(f'\n--> Successfully fetched batch!')
print(f'    Batch Image Tensor Shape : {sample_images.shape} (Batch, Channels, Height, Width)')
print(f'    Batch Labels Tensor Shape: {sample_labels.shape}')
print(f'    Sample Labels            : {sample_labels.tolist()}')
print('\n[READY] The dataset is 100% prepared for training transfer learning models!')
